In [ ]:
import io
import zipfile
import requests
import frontmatter
url = 'https://codeload.github.com/DataTalksClub/faq/zip/refs/heads/main'
resp = requests.get(url)
repository_data = []

# Create a ZipFile object from the downloaded content
zf = zipfile.ZipFile(io.BytesIO(resp.content))

for file_info in zf.infolist():
    filename = file_info.filename.lower()

    # Only process markdown files
    if not filename.endswith('.md'):
        continue

    # Read and parse each file
    with zf.open(file_info) as f_in:
        content = f_in.read()
        post = frontmatter.loads(content)
        data = post.to_dict()
        data['filename'] = filename
        repository_data.append(data)

zf.close()


In [ ]:
print(repository_data[1])

{'content': '# FAQ Bot Feedback - PR Review Corrections\n\n## 1. Wrong Section Placement\n\nKestra-related FAQs were incorrectly placed in `general` or `module-1` instead of `module-2` (workflow orchestration):\n\n| PR | Issue | Correction |\n|----|-------|------------|\n| #141 | Kestra IANA timezones | general → module-2, sort_order 20 |\n| #137 | Kestra stdout variables | general → module-2, sort_order 21 |\n| #135 | Kestra outputFiles visibility | general → module-2, sort_order 22 |\n| #118 | Kestra Docker socket | module-1 → module-2, sort_order 23 |\n\n**Rule**: Kestra questions belong in `module-2` (workflow orchestration), not `general` or `module-1`.\n\n---\n\n## 2. Not Relevant for Course (closed)\n\n| PR | Topic | Reason |\n|----|-------|--------|\n| #123 | Installing vim on Ubuntu | Basic Linux admin, outside course scope |\n| #116 | SQL LEFT JOIN returns NULL | Basic SQL concept, not course-specific |\n\n**Rule**: Fundamental tool/SQL concepts that aren\'t course-specific s

In [ ]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [ ]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"FAQ documents: {len(dtc_faq)}")
print(f"Evidently documents: {len(evidently_docs)}")


FAQ documents: 1285
Evidently documents: 95


In [ ]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result


In [ ]:
evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    evidently_chunks.extend(chunks)


In [ ]:
import re
text = evidently_docs[45]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())


In [ ]:
import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections


In [ ]:
sections = split_markdown_by_level(text, level=2)


In [ ]:
evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)


In [ ]:
evidently_chunks

In [ ]:
%env OPENAI_API_KEY=YOUR_OPENAI_API_KEY

In [ ]:
import os
import certifi
import httpx
from openai import OpenAI

openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError(
        "OPENAI_API_KEY is not set in this notebook kernel. "
        "Set it in a notebook cell with `%env OPENAI_API_KEY=...` "
        "or restart the kernel from a shell where the variable is exported."
    )

# TLS setup for environments with custom/intercepting certificates.
# Set CERT_PATH to your org CA bundle if needed.
ca_bundle = (
    os.getenv("SSL_CERT_FILE")
    or os.getenv("REQUESTS_CA_BUNDLE")
    or os.getenv("CERT_PATH")
    or certifi.where()
)
os.environ["SSL_CERT_FILE"] = ca_bundle
os.environ["REQUESTS_CA_BUNDLE"] = ca_bundle

http_client = httpx.Client(verify=ca_bundle, timeout=60.0)
openai_client = OpenAI(api_key=openai_api_key, http_client=http_client)


def llm(prompt, model="gpt-4o-mini"):
    messages = [{"role": "user", "content": prompt}]

    response = openai_client.responses.create(
        model=model,
        input=messages,
    )

    return response.output_text


In [ ]:
import os
from openai import OpenAI

groq_client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()


In [ ]:
def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)

    try:
        response = llm(prompt)
        sections = response.split('---')
        sections = [s.strip() for s in sections if s.strip()]
        if sections:
            return sections
    except Exception as e:
        # Network/TLS issues should not block notebook progress.
        print("LLM chunking unavailable, using local fallback:", e)

    # Local fallback: split by markdown H2 headers, else fixed windows.
    parts = [p.strip() for p in text.split("\n## ") if p.strip()]
    if len(parts) > 1:
        normalized = []
        for i, p in enumerate(parts):
            if i == 0 and p.startswith("## "):
                normalized.append(p)
            elif i == 0:
                normalized.append(p)
            else:
                normalized.append("## " + p)
        return normalized

    chunk_size = 2000
    step = 1000
    windows = []
    for i in range(0, len(text), step):
        w = text[i:i + chunk_size].strip()
        if w:
            windows.append(w)
        if i + chunk_size >= len(text):
            break
    return windows


In [ ]:
from tqdm.auto import tqdm

evidently_chunks = []

for doc in tqdm(evidently_docs):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')

    sections = intelligent_chunking(doc_content)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)


  0%|          | 0/95 [00:00<?, ?it/s]

LLM chunking unavailable, using local fallback: Connection error.
LLM chunking unavailable, using local fallback: Connection error.
LLM chunking unavailable, using local fallback: Connection error.
LLM chunking unavailable, using local fallback: Connection error.
LLM chunking unavailable, using local fallback: Connection error.
LLM chunking unavailable, using local fallback: Connection error.
LLM chunking unavailable, using local fallback: Connection error.
LLM chunking unavailable, using local fallback: Connection error.
LLM chunking unavailable, using local fallback: Connection error.
LLM chunking unavailable, using local fallback: Connection error.
LLM chunking unavailable, using local fallback: Connection error.
LLM chunking unavailable, using local fallback: Connection error.
LLM chunking unavailable, using local fallback: Connection error.
LLM chunking unavailable, using local fallback: Connection error.
LLM chunking unavailable, using local fallback: Connection error.
LLM chunki

In [ ]:
!pip install tqdm

zsh:1: command not found: pip


In [ ]:
evidently_chunks

In [ ]:
from minsearch import Index

index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

index.fit(evidently_chunks)


In [ ]:
query = 'What should be in a test dataset for AI evaluation?'
results = index.search(query)


In [ ]:
print(results)

In [ ]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')

de_dtc_faq = [d for d in dtc_faq if 'data-engineering' in d['filename']]

faq_index = Index(
    text_fields=["question", "content"],
    keyword_fields=[]
)

faq_index.fit(de_dtc_faq)


In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
record = de_dtc_faq[2]
text = record['question'] + ' ' + record['content']
v_doc = embedding_model.encode(text)


In [ ]:
query = 'I just found out about the course. Can I enroll now?'
v_query = embedding_model.encode(query)


In [ ]:
similarity = v_query.dot(v_doc)


In [ ]:
from tqdm.auto import tqdm
import numpy as np

faq_embeddings = []

for d in tqdm(de_dtc_faq):
    text = d['question'] + ' ' + d['content']
    v = embedding_model.encode(text)
    faq_embeddings.append(v)

faq_embeddings = np.array(faq_embeddings)


  0%|          | 0/498 [00:00<?, ?it/s]

In [ ]:
from minsearch import VectorSearch

faq_vindex = VectorSearch()
faq_vindex.fit(faq_embeddings, de_dtc_faq)


In [ ]:
query = 'Can I join the course now?'
q = embedding_model.encode(query)
results = faq_vindex.search(q)


In [ ]:
results

[{'id': '3f1424af17',
  'question': 'Course: Can I still join the course after the start date?',
  'sort_order': 3,
  'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'},
 {'id': '068529125b',
  'question': 'Course - Can I follow the course after it finishes?',
  'sort_order': 8,
  'content': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.',
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/008_068529125b_course-can-i-follow-the-course-after-i

In [ ]:
evidently_embeddings = []

for d in tqdm(evidently_chunks):
    # Support both legacy ('chunk') and current ('section') record formats.
    text = d.get('chunk') or d.get('section')
    if text is None:
        raise KeyError("Expected 'chunk' or 'section' in evidently_chunks records")

    v = embedding_model.encode(text)
    evidently_embeddings.append(v)

evidently_embeddings = np.array(evidently_embeddings)

evidently_vindex = VectorSearch()
evidently_vindex.fit(evidently_embeddings, evidently_chunks)


  0%|          | 0/378 [00:00<?, ?it/s]

In [ ]:
def text_search(query):
    return faq_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return faq_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)
    
    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)
    
    return combined_results


In [ ]:
query = 'Can I join the course now?'

text_results = faq_index.search(query, num_results=5)

q = embedding_model.encode(query)
vector_results = faq_vindex.search(q, num_results=5)

final_results = text_results + vector_results


In [ ]:
final_results

[{'id': '3f1424af17',
  'question': 'Course: Can I still join the course after the start date?',
  'sort_order': 3,
  'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'},
 {'id': '9e508f2212',
  'question': 'Course: When does the course start?',
  'sort_order': 1,
  'content': "The next cohort starts January 12th, 2026. More info at [DTC](https://datatalks.club/blog/guide-to-free-online-courses-at-datatalks-club.html).\n\n- Register before the course starts using this [link](https://airtable.com/shr6oVXeQvSI5HuWD).\n- Join the [course Telegram channel with announcements](https://t.me/dezoomcamp).\n- Don’t forget to register in DataTalks.Club's Slack and [join](htt

In [ ]:
import os
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("SSL_CERT_FILE:", os.getenv("SSL_CERT_FILE"))
print("REQUESTS_CA_BUNDLE:", os.getenv("REQUESTS_CA_BUNDLE"))
print("CERT_PATH:", os.getenv("CERT_PATH"))

In [ ]:
import openai

openai_client = openai.OpenAI()

user_prompt = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
)

print(response.output_text)


APIConnectionError: Connection error.

In [ ]:
def text_search(query):
    return faq_index.search(query, num_results=5)


In [ ]:
text_search_tool = {
    "type": "function",
    "name": "text_search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}


In [ ]:
system_prompt = """
You are a helpful assistant for a course. 
"""

question = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool]
)


In [ ]:
print(response.output)

In [ ]:
import json

call = response.output[0]

arguments = json.loads(call.arguments)
result = text_search(**arguments)

call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": json.dumps(result),
}


In [ ]:
chat_messages.append(call)
chat_messages.append(call_output)

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool]
)

print(response.output_text)


In [ ]:
chat_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool]
)


In [ ]:
system_prompt = """
You are a helpful assistant for a course. 

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.
If the search doesn't return relevant results, let the user know and provide general guidance.
"""


In [ ]:
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the FAQ index.

    Args:
        query (str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned by the FAQ index.
    """
    return faq_index.search(query, num_results=5)


In [ ]:
import os
import certifi
from pydantic_ai import Agent

# TLS setup for environments with custom/intercepting certificates.
# Priority:
# 1) Existing SSL_CERT_FILE / REQUESTS_CA_BUNDLE if already set.
# 2) CERT_PATH (custom path you set in notebook or shell).
# 3) certifi bundle fallback.
ca_bundle = (
    os.getenv("SSL_CERT_FILE")
    or os.getenv("REQUESTS_CA_BUNDLE")
    or os.getenv("CERT_PATH")
    or certifi.where()
)
os.environ["SSL_CERT_FILE"] = ca_bundle
os.environ["REQUESTS_CA_BUNDLE"] = ca_bundle

agent = Agent(
    name="faq_agent",
    instructions=system_prompt,
    tools=[text_search],
    model="gpt-4o-mini",
)


In [ ]:
question = "I just discovered the course, can I join now?"

try:
    result = await agent.run(user_prompt=question)
    print(result.output)
except Exception as e:
    print("Agent request failed:", e)
    print("SSL_CERT_FILE:", os.getenv("SSL_CERT_FILE"))
    print("If needed, set CERT_PATH to your company CA bundle and re-run.")


In [ ]:
import asyncio

result = asyncio.run(agent.run(user_prompt=question))


In [ ]:
result.new_messages()


In [ ]:
# Day 5: Evaluation (retrieval + answer quality)
import re
import random
import pandas as pd


def _normalize(text: str) -> list[str]:
    text = (text or "").lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return [t for t in text.split() if len(t) > 2]


def token_f1(pred: str, gold: str) -> float:
    pred_tokens = set(_normalize(pred))
    gold_tokens = set(_normalize(gold))
    if not pred_tokens or not gold_tokens:
        return 0.0
    tp = len(pred_tokens & gold_tokens)
    precision = tp / len(pred_tokens)
    recall = tp / len(gold_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def build_course_eval_set(max_items: int = 40, seed: int = 42) -> list[dict]:
    rows = de_dtc_faq.copy()
    rng = random.Random(seed)
    rng.shuffle(rows)

    eval_rows = []
    for row in rows:
        q = (row.get("question") or "").strip()
        a = (row.get("content") or "").strip()
        f = row.get("filename")
        if q and a and f:
            eval_rows.append({
                "question": q,
                "gold_answer": a,
                "gold_filename": f,
            })
        if len(eval_rows) >= max_items:
            break
    return eval_rows


def evaluate_course_retrieval(eval_rows: list[dict], k: int = 5) -> dict:
    hits = 0
    mrr_total = 0.0

    for row in eval_rows:
        results = text_search(row["question"])
        ranked_files = [r.get("filename") for r in results[:k]]

        rank = None
        for i, fname in enumerate(ranked_files, start=1):
            if fname == row["gold_filename"]:
                rank = i
                break

        if rank is not None:
            hits += 1
            mrr_total += 1.0 / rank

    n = max(len(eval_rows), 1)
    return {
        "n": len(eval_rows),
        f"hit@{k}": hits / n,
        "mrr": mrr_total / n,
    }


async def evaluate_course_agent(eval_rows: list[dict], max_items: int = 12) -> dict:
    sample = eval_rows[:max_items]
    f1_scores = []

    for row in sample:
        res = await agent.run(user_prompt=row["question"])
        pred = res.output if hasattr(res, "output") else str(res)
        f1_scores.append(token_f1(pred, row["gold_answer"]))

    n = max(len(f1_scores), 1)
    return {
        "n": len(f1_scores),
        "avg_token_f1": sum(f1_scores) / n,
    }


eval_rows = build_course_eval_set(max_items=40)
retrieval_report = evaluate_course_retrieval(eval_rows, k=5)
agent_report = await evaluate_course_agent(eval_rows, max_items=12)

pd.DataFrame([retrieval_report, agent_report])